# Run a Monte Carlo experiment · Acadia National Park, Maine (DEM)

Runs every **condition** with every **seed**, in parallel. A condition is a set of config overrides with dotted
keys (e.g. `"robot.exploration_map"`). Each time a run finishes, the comparison plot below is redrawn from the runs
completed so far. Results go to `OUT_DIR/<condition>/seed_<n>/`; open
`notebooks/analysis/montecarlo_report.ipynb` on `OUT_DIR` for the full report.

To watch a single run step by step, use `run_simulation.ipynb` with the same configuration.

> Known issue: about 68% of this map is water and the home base sits on a water cell, so agents barely leave the water (0 harvests in a 30-day check). Revisit the DEM stream threshold and the home base in the data pipeline.

In [ ]:
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "beaversim").is_dir())
sys.path.insert(0, str(root))

from beaversim import viz
from beaversim.experiments import run, run_montecarlo, apply_overrides

## Parameters

In [ ]:
OUT_DIR = root / "output/simulations/acadia/exploration_map"
SEEDS = range(1, 6)
PROCESSES = 10               # parallel runs (this machine has 24 cores)
MONITOR_METRIC = "diffusion" # redrawn after each run: "diffusion", "canalization", "water", "vegetation"
MEDIUM = "chapter"

CONDITIONS = {
    "quality": {"robot.exploration_map": ["vegetation_quality", "vegetation_quality"]},
    "visits":  {"robot.exploration_map": ["vegetation_visits", "vegetation_visits"]},
}

## Base configuration
Shared by every condition; see `run_simulation.ipynb` for the key reference.

In [ ]:
SITE = root / "output/Acadia_Maine/DEM"     # prepared site (data_acquisition pipeline output)

config = {
    "backend_type": "beavers_visualizer",
    "environment": {
        "name": "beavers_environment",
        "elevation_file_path": str(SITE / "elevation.npy"),
        "metadata_file_path": str(SITE / "processing_metadata.json"),
        # Vegetation
        "vegetation_quality_range": [0, 10],
        "grass_growth_interval": [0, 8],
        "grass_growth_rate": 5e-4,
        "grass_growth_sigma": 0.5,
        "mean_reversion_strength": 0.0005,
        # Stigmergy
        "visits_reset": 0.99,
        # Water
        "flow_info": {
            "river_growth_velocity": 1e-4,
            "river_growth_interval": [-3, 0],
            "streams_depth": 10,
            "direction": [1, -1],
            "strength": 0.25,
        },
        "print": False,
    },
    "robot": {
        "home_base_position": [[50, 45]],                  # metres; one or more lodges
        "role": ["explorer", "builder"],                   # each agent draws one at random
        # Per-role parameters: [explorer, builder]
        "maximum_load": [1000, 10],
        "harvest_interval": [[4, 8], [0, 2]],
        "epsilon_greedy": [0.1, 0.9],
        "exploration_map": ["vegetation_quality", "vegetation_visits"],
        "vegetation_removal": [0.1, 0.1],
        # Motion
        "controller": {
            "name": "P_repulsive",
            "Kp": 0.5, "Kd": 2.0, "Ki": 1e-5,
            "neighbourhood_size": 2,
            "accuracy": 0.5,
            "map_repulsive": "vegetation_quality",
            "beta_repulsive": 0.2,
        },
        "dynamics": {"name": "integrator", "mass": 5, "friction": 1},
        # Stigmergy and adaptation
        "visit_increase": 1.0,
        "decay_values": [0.05, 0.05],                    # [maximum_load decay, harvest_interval widening]
        "print": False,
    },
    "simulation": {
        "seed": 1,
        "timedelta": 1.0,                                  # hours per step
        "schedule_policy": "sequential",
        "number_of_agents": 5,
        "number_of_steps": 365,                            # duration in DAYS
        "downsampling": 15,                                # days between saved snapshots
        "print": False,
    },
}

## Run

In [ ]:
viz.use_theme(MEDIUM)
run_montecarlo(config, OUT_DIR, seeds=SEEDS, conditions=CONDITIONS, processes=PROCESSES,
               on_done=viz.live.MonteCarloMonitor(OUT_DIR, MONITOR_METRIC))
print("Experiment recorded in", OUT_DIR)

## Next
Open `notebooks/analysis/montecarlo_report.ipynb` and set `EXPERIMENT_DIR` to the folder printed above.